#  Electrolyte Box Equilibrations

## Configurations

> Initial configurations were packed with Packmol and geometry-optimized with the FIRE algorithm. Systems were then equilibrated in two stages: NVT Langevin dynamics (50 ps, 300 K) to thermalize the structure, followed by NPT dynamics (100 ps, 300 K, 1 atm) to relax the box volume to equilibrium density. Both stages used the OrbMol-v2 potential (Orbital Materials), trained on the OMol25 dataset.

## How to use this

1. Run section 1 once per session. Installs packages, mounts Drive, defines the functions.
2. Section 2 is NVT thermalization. It has to finish before NPT.
3. Section 3 is NPT density equilibration.
4. Section 4 is the status check and download.

If the runtime drops, reconnect, rerun section 1, then use the resume cell (2b or 3c) to pick up from the last checkpoint.

One box at a time. When one finishes, come back and start the next by hand.

## 1.) Installs and setup

In [ ]:
!pip install -q ase orb-models

In [ ]:
import os, glob, json, time
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.melchionna import MelchionnaNPT as NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
import torch

from orb_models.forcefield import pretrained
from orb_models.forcefield.inference.calculator import ORBCalculator

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

DRIVE_ROOT = '/content/gdrive/MyDrive/Dataset1_OrbMolV2_equilibration'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
PRESSURE_EV_A3 = 0.000101325 * 0.006242   # 1 atm in eV/A^3
BULK_MOD_EV_A3 = 1.0 * 0.006242           # ~1 GPa for organic liquids
PFACTOR = (75.0 * units.fs)**2 * BULK_MOD_EV_A3
NVT_STEPS = 50000    # 50 ps
NPT_STEPS = 100000   # 100 ps
CHECKPOINT_EVERY = 5000
LOG_EVERY = 100


def get_calculator(device='cuda'):
    """Load OrbMol-v2 (OMol25-trained) as an ASE calculator."""
    orbff, atoms_adapter = pretrained.orbmol_v2(
        device=device, precision='float32-high'
    )
    return ORBCalculator(orbff, atoms_adapter=atoms_adapter, device=device)


def box_dir(name):
    d = os.path.join(DRIVE_ROOT, name)
    os.makedirs(d, exist_ok=True)
    return d


def save_checkpoint(atoms, name, step, phase, temps, pes, densities=None):
    """Save to Drive, deleting the older ones so only the latest survives."""
    d = box_dir(name)
    for f in glob.glob(os.path.join(d, 'ckpt_*.xyz')):
        os.remove(f)
    for f in glob.glob(os.path.join(d, 'ckpt_*.npz')):
        os.remove(f)
    write(os.path.join(d, f'ckpt_{phase}_{step}.xyz'), atoms, format='extxyz')
    save_data = dict(step=step, phase=phase,
                     temps=np.array(temps), pes=np.array(pes))
    if densities is not None:
        save_data['densities'] = np.array(densities)
    np.savez(os.path.join(d, f'ckpt_{phase}_{step}.npz'), **save_data)
    rho_str = f', rho={densities[-1]:.4f}' if densities else ''
    print(f'    [CHECKPOINT] {phase} step {step}{rho_str} -> {d}', flush=True)


def load_checkpoint(name):
    """Load latest checkpoint. Returns (atoms, step, phase, temps, pes, densities)."""
    d = box_dir(name)
    xyzs = sorted(glob.glob(os.path.join(d, 'ckpt_*.xyz')))
    npzs = sorted(glob.glob(os.path.join(d, 'ckpt_*.npz')))
    if not xyzs or not npzs:
        return None, 0, None, [], [], []
    atoms = read(xyzs[-1])
    data = np.load(npzs[-1], allow_pickle=True)
    densities = data['densities'].tolist() if 'densities' in data else []
    return (atoms, int(data['step']), str(data['phase']),
            data['temps'].tolist(), data['pes'].tolist(), densities)


def run_nvt(atoms, name, n_steps=NVT_STEPS, start_step=0,
            prev_temps=None, prev_pes=None):
    """Run NVT Langevin dynamics with checkpointing."""
    atoms.pbc = True
    atoms.info['charge'] = 0
    atoms.info['spin'] = 1

    torch.cuda.empty_cache()
    atoms.calc = get_calculator()

    if start_step == 0:
        MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)

    dyn = Langevin(atoms, timestep=1.0*units.fs, temperature_K=300.0,
                   friction=0.01/units.fs)
    dyn.nsteps = start_step

    temps = list(prev_temps or [])
    pes = list(prev_pes or [])
    remaining = n_steps - start_step
    t0 = time.time()

    print(f'  {len(atoms)} atoms | {remaining} steps remaining', flush=True)

    def logger():
        T = atoms.get_temperature()
        pe = atoms.get_potential_energy() / len(atoms)
        temps.append(T); pes.append(pe)
        step = dyn.nsteps
        if step % 1000 == 0:
            elapsed = time.time() - t0
            rate = (step - start_step) / elapsed if elapsed > 0 else 0
            eta = (n_steps - step) / rate / 60 if rate > 0 else 0
            print(f'    step {step:6d}/{n_steps}  T={T:.1f}K  '
                  f'PE={pe:.4f}  [{rate:.1f} st/s, ETA {eta:.0f}m]', flush=True)

    def checkpointer():
        step = dyn.nsteps
        if step > start_step and step % CHECKPOINT_EVERY == 0:
            save_checkpoint(atoms, name, step, 'nvt', temps, pes)

    dyn.attach(logger, interval=LOG_EVERY)
    dyn.attach(checkpointer, interval=LOG_EVERY)

    if remaining <= 0:
        print(f'  Already complete.', flush=True)
        return atoms, temps, pes

    dyn.run(remaining)
    save_checkpoint(atoms, name, n_steps, 'nvt', temps, pes)
    return atoms, temps, pes


def run_npt(atoms, name, n_steps=NPT_STEPS, start_step=0,
            prev_temps=None, prev_pes=None, prev_densities=None):
    """Run NPT dynamics with checkpointing and density tracking."""
    atoms.pbc = True
    atoms.info['charge'] = 0
    atoms.info['spin'] = 1
    total_mass = sum(atoms.get_masses())

    torch.cuda.empty_cache()
    atoms.calc = get_calculator()

    if start_step == 0:
        MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)

    dyn = NPT(atoms, timestep=1.0*units.fs, temperature_K=300.0,
              externalstress=PRESSURE_EV_A3, ttime=25*units.fs, pfactor=PFACTOR)
    dyn.nsteps = start_step

    temps = list(prev_temps or [])
    pes = list(prev_pes or [])
    densities = list(prev_densities or [])
    remaining = n_steps - start_step
    t0 = time.time()

    rho0 = total_mass / atoms.get_volume() * 1.66054
    print(f'  {len(atoms)} atoms | starting rho={rho0:.4f} g/cm3 | {remaining} steps remaining', flush=True)

    def logger():
        T = atoms.get_temperature()
        pe = atoms.get_potential_energy() / len(atoms)
        rho = total_mass / atoms.get_volume() * 1.66054
        temps.append(T); pes.append(pe); densities.append(rho)
        step = dyn.nsteps
        if step % 1000 == 0:
            elapsed = time.time() - t0
            rate = (step - start_step) / elapsed if elapsed > 0 else 0
            eta = (n_steps - step) / rate / 60 if rate > 0 else 0
            print(f'    step {step:6d}/{n_steps}  T={T:.1f}K  rho={rho:.4f}  '
                  f'PE={pe:.4f}  [{rate:.1f} st/s, ETA {eta:.0f}m]', flush=True)

    def checkpointer():
        step = dyn.nsteps
        if step > start_step and step % CHECKPOINT_EVERY == 0:
            save_checkpoint(atoms, name, step, 'npt', temps, pes, densities)

    dyn.attach(logger, interval=LOG_EVERY)
    dyn.attach(checkpointer, interval=LOG_EVERY)

    if remaining <= 0:
        print(f'  Already complete.', flush=True)
        return atoms, temps, pes, densities

    dyn.run(remaining)

    save_checkpoint(atoms, name, n_steps, 'npt', temps, pes, densities)
    final_path = os.path.join(box_dir(name), f'{name}.xyz')
    write(final_path, atoms, format='extxyz')

    rho_f = total_mass / atoms.get_volume() * 1.66054
    has_nan = np.any(np.isnan(atoms.get_positions()))
    if has_nan:
        print(f'  WARNING: NaN in final positions!', flush=True)
    else:
        print(f'  DONE: rho {rho0:.4f} -> {rho_f:.4f} g/cm3', flush=True)
        print(f'  Final: {final_path}', flush=True)

    return atoms, temps, pes, densities


def save_final(atoms, name):
    d = box_dir(name)
    path = os.path.join(d, f'{name}.xyz')
    write(path, atoms, format='extxyz')
    if np.any(np.isnan(atoms.get_positions())):
        print(f'  WARNING: NaN in final positions for {name}!')
    else:
        print(f'  Final structure: {path}')
    return path


def plot_diagnostics(name, temps, pes, densities=None):
    if len(temps) == 0:
        print('No data to plot.')
        return
    n_plots = 3 if densities else 2
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 4))
    t_ps = np.arange(len(temps)) * LOG_EVERY * 0.001
    w = min(50, max(1, len(temps) // 4))

    axes[0].plot(t_ps, temps, alpha=0.3, lw=0.5)
    if w > 1: axes[0].plot(t_ps[w-1:], np.convolve(temps, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    axes[0].axhline(300, color='k', ls='--', alpha=0.4)
    axes[0].set(xlabel='Time (ps)', ylabel='Temperature (K)', title='Temperature')

    axes[1].plot(t_ps, pes, alpha=0.3, lw=0.5)
    if w > 1: axes[1].plot(t_ps[w-1:], np.convolve(pes, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    axes[1].set(xlabel='Time (ps)', ylabel='PE (eV/atom)', title='Potential Energy')

    if densities and len(densities) > 0:
        t_rho = np.arange(len(densities)) * LOG_EVERY * 0.001
        axes[2].plot(t_rho, densities, alpha=0.3, lw=0.5)
        if w > 1: axes[2].plot(t_rho[w-1:], np.convolve(densities, np.ones(w)/w, 'valid'), 'r', lw=1.5)
        axes[2].set(xlabel='Time (ps)', ylabel='Density (g/cm3)', title='Density')

    plt.suptitle(name, fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(box_dir(name), 'diagnostics.png'), dpi=150)
    plt.show()

print('Setup complete. OrbMol-v2 ready.')

## 2.) NVT thermalization

Langevin at 300 K to thermalize velocities and relax the local structure. Has to finish before NPT.

### 2a. Start NVT from scratch (upload a box)

Upload a FIRE-optimized `.xyz` file. The box name comes from the filename.

In [ ]:
from google.colab import files
print('Upload the FIRE-optimized .xyz file to start NVT:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
BOX_NAME = INPUT_FILE.replace('_opt.xyz', '').replace('.xyz', '')
print(f'\nBox: {BOX_NAME}')
print(f'Checkpoints: {DRIVE_ROOT}/{BOX_NAME}/')

In [ ]:
atoms = read(INPUT_FILE)
print(f'Loaded: {len(atoms)} atoms, cell = {atoms.cell.lengths()}')

print(f'\nNVT thermalization: {BOX_NAME}')
atoms, temps, pes = run_nvt(atoms, BOX_NAME)

final_path = save_final(atoms, BOX_NAME)
plot_diagnostics(BOX_NAME, temps, pes)
print(f'\nNVT complete. Final structure: {final_path}')

### 2b. Resume NVT from checkpoint

If the runtime dropped during NVT, reconnect, run section 1, then run this.

In [ ]:
print('Boxes with checkpoints on Drive:\n')
box_dirs = sorted([d for d in os.listdir(DRIVE_ROOT) if os.path.isdir(os.path.join(DRIVE_ROOT, d))])
for i, d in enumerate(box_dirs):
    dd = os.path.join(DRIVE_ROOT, d)
    ckpts = sorted(glob.glob(os.path.join(dd, 'ckpt_*.xyz')))
    final = os.path.exists(os.path.join(dd, f'{d}.xyz'))
    status = 'FINISHED' if final else (f'latest: {os.path.basename(ckpts[-1])}' if ckpts else 'no checkpoints')
    print(f'  [{i}] {d:<50s} {status}')
print('\nSet RESUME_INDEX below, then run the next cell.')

In [ ]:
RESUME_INDEX = 0  # <-- change this

resume_name = box_dirs[RESUME_INDEX]
atoms, step, phase, prev_t, prev_pe, prev_rho = load_checkpoint(resume_name)

if atoms is None:
    print(f'No checkpoint for {resume_name}. Use 2a instead.')
elif phase == 'npt':
    print(f'This box is already in NPT. Use Section 3c to resume NPT.')
elif phase == 'nvt' and step >= NVT_STEPS:
    print(f'NVT already complete at step {step}. Proceed to NPT (Section 3).')
else:
    print(f'Resuming NVT: {resume_name} from step {step}')
    atoms.pbc = True
    atoms, temps, pes = run_nvt(atoms, resume_name, start_step=step,
                                prev_temps=prev_t, prev_pes=prev_pe)
    final_path = save_final(atoms, resume_name)
    plot_diagnostics(resume_name, temps, pes)
    print(f'NVT complete. Final structure: {final_path}')

## 3.) NPT density equilibration

NPT at 300 K, 1 atm to let the box volume settle to the right density. NVT has to be done first.

### 3a. Start NPT from scratch (upload a box)

In [ ]:
from google.colab import files
print('Upload one NVT-equilibrated .xyz file:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
BOX_NAME = INPUT_FILE.replace('.xyz', '')
print(f'\nBox: {BOX_NAME}')
print(f'Checkpoints: {DRIVE_ROOT}/{BOX_NAME}/')

In [ ]:
atoms = read(INPUT_FILE)
print(f'\nNPT: {BOX_NAME}')

atoms, temps, pes, densities = run_npt(atoms, BOX_NAME)
plot_diagnostics(BOX_NAME, temps, pes, densities)

### 3b. Start NPT from a box already on Drive

In [ ]:
drive_boxes = sorted([f.replace('.xyz','') for f in os.listdir(DRIVE_ROOT)
                       if f.endswith('.xyz') and not f.startswith('ckpt_')])
for d in sorted(os.listdir(DRIVE_ROOT)):
    dd = os.path.join(DRIVE_ROOT, d)
    if os.path.isdir(dd):
        inp = os.path.join(dd, f'{d}_input.xyz')
        final = os.path.join(dd, f'{d}.xyz')
        if os.path.exists(inp) and not os.path.exists(final):
            drive_boxes.append(d + ' (has input, not finished)')

print('Boxes available on Drive:')
for i, b in enumerate(drive_boxes):
    print(f'  [{i}] {b}')
print('\nSet BOX_INDEX below, then run the next cell.')

In [ ]:
BOX_INDEX = 0  # <-- change this
BOX_NAME = drive_boxes[BOX_INDEX].split(' (')[0]

candidates = [os.path.join(DRIVE_ROOT, f'{BOX_NAME}.xyz'),
              os.path.join(DRIVE_ROOT, BOX_NAME, f'{BOX_NAME}_input.xyz')]
INPUT_FILE = next((c for c in candidates if os.path.exists(c)), None)
if INPUT_FILE is None:
    print(f'No input file for {BOX_NAME}. Upload with 3a instead.')
else:
    atoms = read(INPUT_FILE)
    inp_copy = os.path.join(box_dir(BOX_NAME), f'{BOX_NAME}_input.xyz')
    if not os.path.exists(inp_copy):
        write(inp_copy, atoms, format='extxyz')
    print(f'\nNPT: {BOX_NAME}')
    atoms, temps, pes, densities = run_npt(atoms, BOX_NAME)
    plot_diagnostics(BOX_NAME, temps, pes, densities)

### 3c. Resume NPT from checkpoint

If the runtime dropped mid-NPT, reconnect, run section 1, then use this.

In [ ]:
print('Boxes on Drive:\n')
all_dirs = sorted([d for d in os.listdir(DRIVE_ROOT) if os.path.isdir(os.path.join(DRIVE_ROOT, d))])
for i, d in enumerate(all_dirs):
    dd = os.path.join(DRIVE_ROOT, d)
    final = os.path.exists(os.path.join(dd, f'{d}.xyz'))
    ckpts = sorted(glob.glob(os.path.join(dd, 'ckpt_*.npz')))
    if final:
        status = 'FINISHED'
    elif ckpts:
        data = np.load(ckpts[-1], allow_pickle=True)
        status = f'{str(data["phase"])} step {int(data["step"])}'
    else:
        status = 'no checkpoint'
    print(f'  [{i}] {d:<50s} {status}')
print('\nSet RESUME_INDEX below, then run the next cell.')

In [ ]:
RESUME_INDEX = 0  # <-- change this

resume_name = all_dirs[RESUME_INDEX]
atoms, step, phase, prev_t, prev_pe, prev_rho = load_checkpoint(resume_name)

if atoms is None:
    print(f'No checkpoint for {resume_name}. Use 3a to start fresh.')
elif phase == 'nvt':
    print(f'This box is still in NVT phase. Use Section 2b to finish NVT first.')
elif step >= NPT_STEPS:
    print(f'Already complete! Use Section 4 to download.')
else:
    print(f'Resuming NPT: {resume_name} from step {step}')
    atoms.pbc = True
    atoms, temps, pes, densities = run_npt(
        atoms, resume_name, start_step=step,
        prev_temps=prev_t, prev_pes=prev_pe, prev_densities=prev_rho)
    plot_diagnostics(resume_name, temps, pes, densities)

## 4.) Status and download

In [ ]:
print(f'{"Box":<50s} {"Status":<15s} {"Final rho":<12s} {"Atoms"}')
print('-' * 90)
finished = []
for d in sorted(os.listdir(DRIVE_ROOT)):
    dd = os.path.join(DRIVE_ROOT, d)
    if not os.path.isdir(dd): continue
    final = os.path.join(dd, f'{d}.xyz')
    if os.path.exists(final):
        a = read(final)
        has_nan = np.any(np.isnan(a.get_positions()))
        if has_nan:
            status, rho_str = 'NaN!', 'N/A'
        else:
            rho = sum(a.get_masses()) / a.get_volume() * 1.66054
            status, rho_str = 'DONE', f'{rho:.4f} g/cm3'
            finished.append((d, final, rho))
        print(f'{d:<50s} {status:<15s} {rho_str:<12s} {len(a)}')
    else:
        ckpts = sorted(glob.glob(os.path.join(dd, 'ckpt_*.npz')))
        if ckpts:
            data = np.load(ckpts[-1], allow_pickle=True)
            rho = data['densities'][-1] if 'densities' in data and len(data['densities']) > 0 else 0
            status = f'{str(data["phase"])} step {int(data["step"])}'
            print(f'{d:<50s} {status:<15s} {rho:.4f} g/cm3' if rho else f'{d:<50s} {status:<15s}')
        else:
            print(f'{d:<50s} {"empty":<15s}')

print(f'\n{len(finished)} boxes finished.')

In [ ]:
import shutil

if finished:
    dl_dir = '/content/npt_finished'
    os.makedirs(dl_dir, exist_ok=True)
    for name, path, rho in finished:
        shutil.copy(path, os.path.join(dl_dir, f'{name}.xyz'))
        diag = os.path.join(os.path.dirname(path), 'diagnostics.png')
        if os.path.exists(diag):
            shutil.copy(diag, os.path.join(dl_dir, f'{name}_diagnostics.png'))
    shutil.make_archive('/content/npt_finished', 'zip', dl_dir)
    from google.colab import files
    files.download('/content/npt_finished.zip')
    print(f'Downloaded {len(finished)} boxes as npt_finished.zip')
else:
    print('No finished boxes to download.')